# Batch LLM Inference with Ray Data

This notebook demonstrates how to perform efficient offline batch inference on large datasets using Ray Data and vLLM. You'll learn how to:

- Process large datasets with Ray Data streaming pipelines
- Optimize CPU-to-GPU data flow for maximum throughput
- Run batch inference with vLLM for high efficiency
- Handle embeddings, classifications, and text generation at scale

## When to Use Batch Inference

Use batch inference (not real-time serving) when you need to:
- Process large datasets offline
- Generate embeddings for a document corpus
- Classify thousands of documents
- Create synthetic data or augment datasets

## Prerequisites

- 1+ GPUs
- Python 3.9+

## 1. Installation

In [ ]:
!pip install -q "ray[data]" vllm transformers datasets pandas

## 2. Setup

In [ ]:
import os
import sys

sys.path.insert(0, ".")

from utils import (
    print_gpu_status,
    detect_gpus,
    init_ray,
    ClusterMode,
)

In [ ]:
print_gpu_status()

gpu_info = detect_gpus()
NUM_GPUS = gpu_info["count"] if gpu_info["available"] else 0
print(f"\nAvailable GPUs: {NUM_GPUS}")

In [ ]:
# Configuration
CLUSTER_MODE = ClusterMode.LOCAL

# Model for inference
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Batch processing config
BATCH_SIZE = 32  # Texts per batch
NUM_GPU_ACTORS = max(1, NUM_GPUS)  # One actor per GPU
CONCURRENCY = NUM_GPU_ACTORS  # Concurrent inference tasks

# Output
OUTPUT_PATH = "./runs/batch_inference/results"

In [ ]:
init_ray(mode=CLUSTER_MODE)

## 3. Prepare Dataset

Ray Data supports various input formats: CSV, JSON, Parquet, HuggingFace datasets, and more.

In [ ]:
import ray
from datasets import load_dataset

# Load a sample dataset
hf_dataset = load_dataset("ag_news", split="test[:1000]")  # 1000 samples for demo

# Convert to Ray Dataset
ray_dataset = ray.data.from_huggingface(hf_dataset)

print(f"Dataset size: {ray_dataset.count()} samples")
print(f"Schema: {ray_dataset.schema()}")
print(f"\nSample:")
ray_dataset.take(1)

## 4. Define Inference Class

Ray Data uses "map" operations with callable classes for GPU-based inference.

In [ ]:
from typing import Dict, List
import numpy as np


class LLMPredictor:
    """
    Stateful predictor class for batch inference.
    
    Ray Data will instantiate this class once per GPU actor,
    so the model is loaded only once and reused for all batches.
    """
    
    def __init__(self, model_name: str, task: str = "generation"):
        """
        Initialize the predictor with a model.
        
        Args:
            model_name: HuggingFace model name
            task: "generation", "classification", or "embedding"
        """
        from vllm import LLM, SamplingParams
        
        self.task = task
        self.model_name = model_name
        
        # Initialize vLLM for generation tasks
        if task in ["generation", "classification"]:
            self.llm = LLM(
                model=model_name,
                trust_remote_code=True,
                gpu_memory_utilization=0.9,
            )
            self.sampling_params = SamplingParams(
                temperature=0.1,  # Low temp for deterministic outputs
                max_tokens=100,
            )
    
    def __call__(self, batch: Dict[str, np.ndarray]) -> Dict[str, np.ndarray]:
        """
        Process a batch of inputs.
        
        Args:
            batch: Dictionary with column names as keys and numpy arrays as values
        
        Returns:
            Dictionary with original data plus predictions
        """
        texts = batch["text"].tolist()
        
        if self.task == "generation":
            predictions = self._generate(texts)
        elif self.task == "classification":
            predictions = self._classify(texts)
        else:
            raise ValueError(f"Unknown task: {self.task}")
        
        # Return batch with predictions added
        return {
            **batch,
            "prediction": np.array(predictions),
        }
    
    def _generate(self, texts: List[str]) -> List[str]:
        """Generate text completions."""
        # Format prompts
        prompts = [f"Summarize this news article in one sentence:\n\n{text}\n\nSummary:" for text in texts]
        
        # Run inference
        outputs = self.llm.generate(prompts, self.sampling_params)
        
        return [output.outputs[0].text.strip() for output in outputs]
    
    def _classify(self, texts: List[str]) -> List[str]:
        """Classify texts into categories."""
        categories = ["World", "Sports", "Business", "Science/Tech"]
        
        prompts = [
            f"""Classify this news article into one of these categories: {', '.join(categories)}

Article: {text}

Category:"""
            for text in texts
        ]
        
        outputs = self.llm.generate(prompts, self.sampling_params)
        
        # Extract category from output
        predictions = []
        for output in outputs:
            pred = output.outputs[0].text.strip().split()[0].rstrip(",.:")
            # Validate against categories
            if pred not in categories:
                pred = "Unknown"
            predictions.append(pred)
        
        return predictions

## 5. Run Batch Generation

In [ ]:
# Run batch inference with text generation
print(f"Running batch inference with {NUM_GPU_ACTORS} GPU actor(s)...")
print(f"Batch size: {BATCH_SIZE}")
print(f"Model: {MODEL_NAME}")
print("="*50)

# Map the predictor across the dataset
# Ray Data handles:
# - Parallel execution across GPUs
# - Automatic batching
# - Memory management
# - Streaming results

results_ds = ray_dataset.map_batches(
    LLMPredictor,
    fn_constructor_kwargs={"model_name": MODEL_NAME, "task": "generation"},
    batch_size=BATCH_SIZE,
    num_gpus=1,  # GPUs per actor
    concurrency=CONCURRENCY,  # Number of parallel actors
)

# Show some results
print("\nSample results:")
for row in results_ds.take(3):
    print(f"Original: {row['text'][:100]}...")
    print(f"Summary: {row['prediction']}")
    print("-"*50)

## 6. Run Batch Classification

In [ ]:
# Run batch classification
print("Running batch classification...")
print("="*50)

classification_ds = ray_dataset.map_batches(
    LLMPredictor,
    fn_constructor_kwargs={"model_name": MODEL_NAME, "task": "classification"},
    batch_size=BATCH_SIZE,
    num_gpus=1,
    concurrency=CONCURRENCY,
)

# Show classification results
print("\nClassification results:")
for row in classification_ds.take(5):
    actual_label = ["World", "Sports", "Business", "Science/Tech"][row["label"]]
    print(f"Text: {row['text'][:80]}...")
    print(f"Predicted: {row['prediction']}, Actual: {actual_label}")
    print("-"*50)

## 7. Save Results

In [ ]:
import os

# Create output directory
os.makedirs(OUTPUT_PATH, exist_ok=True)

# Save to Parquet (efficient for large datasets)
results_ds.write_parquet(f"{OUTPUT_PATH}/summaries")
print(f"Saved summaries to {OUTPUT_PATH}/summaries")

classification_ds.write_parquet(f"{OUTPUT_PATH}/classifications")
print(f"Saved classifications to {OUTPUT_PATH}/classifications")

In [ ]:
# Alternatively, save to JSON for readability
results_ds.write_json(f"{OUTPUT_PATH}/summaries_json")
print(f"Saved JSON to {OUTPUT_PATH}/summaries_json")

## 8. Advanced: CPU Preprocessing Pipeline

For optimal throughput, Ray Data streams data from CPU preprocessing to GPU inference.

In [ ]:
def preprocess_text(batch: Dict[str, np.ndarray]) -> Dict[str, np.ndarray]:
    """
    CPU-based preprocessing step.
    
    This runs on CPU workers and streams data to GPU workers.
    """
    texts = batch["text"]
    
    # Clean and normalize text
    cleaned = []
    for text in texts:
        # Remove extra whitespace
        text = " ".join(text.split())
        # Truncate very long texts
        if len(text) > 1000:
            text = text[:1000] + "..."
        cleaned.append(text)
    
    return {
        **batch,
        "text": np.array(cleaned),
        "text_length": np.array([len(t) for t in cleaned]),
    }


# Full pipeline: CPU preprocess -> GPU inference
pipeline = (
    ray_dataset
    # CPU preprocessing (runs on CPU workers)
    .map_batches(
        preprocess_text,
        batch_size=BATCH_SIZE * 2,  # Larger batches for CPU
        num_cpus=1,
    )
    # GPU inference (streams from CPU preprocessing)
    .map_batches(
        LLMPredictor,
        fn_constructor_kwargs={"model_name": MODEL_NAME, "task": "generation"},
        batch_size=BATCH_SIZE,
        num_gpus=1,
        concurrency=CONCURRENCY,
    )
)

print("Pipeline created. Executing...")
print(f"Sample output: {pipeline.take(1)}")

## 9. Performance Monitoring

In [ ]:
import time

# Time a full pipeline execution
start_time = time.time()

# Materialize the entire dataset
count = results_ds.count()

elapsed = time.time() - start_time
throughput = count / elapsed

print(f"Processed {count} samples in {elapsed:.2f} seconds")
print(f"Throughput: {throughput:.1f} samples/second")

## 10. Cleanup

In [ ]:
from utils import shutdown_ray
shutdown_ray()

## Key Takeaways

1. **Ray Data + vLLM**: Combines streaming data pipelines with high-throughput inference
2. **CPU-GPU Streaming**: Preprocessing happens on CPU while GPU runs inference
3. **Automatic Batching**: Ray Data handles optimal batch sizes for your hardware
4. **Scalability**: Same code works from 1 GPU to hundreds

## Next Steps

- **Embeddings**: Use sentence-transformers for embedding generation
- **Multi-modal**: Process images + text with Ray Data
- **Production**: Deploy as a scheduled batch job on Anyscale

## Resources

- [Ray Data Documentation](https://docs.ray.io/en/latest/data/data.html)
- [Working with LLMs in Ray Data](https://docs.ray.io/en/latest/data/working-with-llms.html)
- [vLLM Offline Inference](https://docs.vllm.ai/en/latest/getting_started/quickstart.html)